# Estimativas e tamanhos de efeito

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/01_estimativas_e_tamanhos_de_efeito_revisado.ipynb)

A Unidade 5 introduziu a ideia de **agrupamento**: o mesmo dataset pode ser dividido segundo critérios diferentes, e cada agrupamento permite revelar propriedades entre subconjuntos de dados.

Neste notebook começaremos com dois grupos definidos por `local` — `Capital` e `Interior` — e uma propriedade numérica, `palavras`.

A pergunta inicial é:

> **Quanto os grupos diferem no conjunto observado?**

Primeiro descrevemos e quantificamos a diferença. Incerteza e testes de hipótese serão tratados depois.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os, subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/dados/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_05"

try:
    import google.colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git","clone","--depth","1","--branch","main",URL_REPOSITORIO,str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /dados/disciplina_computacao_aplicada_humanidades_digitais/unidade_05


## 1. Dados, grupos e quantidade de interesse

Cada linha representa um documento fictício. A versão revisada usa o conjunto mais amplo da Unidade 5: **120 documentos**, com `ano`, `genero`, `local`, `tema`, `palavras`, `paginas` e `pessoas`.

O agrupamento inicial é:

\[
G_C=\{d: local(d)=\text{Capital}\},
\qquad
G_I=\{d: local(d)=\text{Interior}\}.
\]

Esse agrupamento é uma **escolha analítica**. Os mesmos registros poderiam ser agrupados por gênero, tema ou período.

Também precisamos declarar **qual propriedade coletiva** queremos comparar. Para extensão documental, por exemplo:

- a média usa todos os valores;
- a mediana descreve a posição central com menor sensibilidade a extremos;
- uma proporção responde a uma pergunta categórica.

Se \(\hat\theta_A\) e \(\hat\theta_B\) são estimativas nos grupos, a diferença absoluta é

\[
\Delta=\hat\theta_A-\hat\theta_B.
\]

O sinal depende da ordem, que deve sempre ser declarada.

In [2]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARQUIVO = Path("dados/documentos.csv")

def gerar_dados():
    SEED, N = 20261058, 120
    rng = np.random.default_rng(SEED)

    linhas = []
    for ano in range(1890, 1910):
        t = (ano - 1890) / 19
        p_capital = 0.12 + 0.76 * t
        n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
        locais_ano = np.array(["Capital"]*n_capital + ["Interior"]*(6-n_capital))
        rng.shuffle(locais_ano)
        linhas.extend((ano, local) for local in locais_ano)

    rng.shuffle(linhas)
    anos = np.array([x[0] for x in linhas])
    locais = np.array([x[1] for x in linhas])
    capital = locais == "Capital"

    generos = rng.choice(["notícia","editorial","carta"], N, p=[0.45,0.30,0.25])

    temas_possiveis = np.array(["educação","trabalho","progresso","saúde"])
    probs = {
        "notícia":[0.20,0.25,0.15,0.40],
        "editorial":[0.20,0.30,0.35,0.15],
        "carta":[0.40,0.25,0.20,0.15],
    }
    temas = np.array([rng.choice(temas_possiveis,p=probs[g]) for g in generos])

    efeito_genero = {"notícia":0.03,"editorial":0.24,"carta":-0.18}
    log_palavras = (
        math.log(650)
        + np.array([efeito_genero[g] for g in generos])
        + np.where(capital,0.64,0.0)
        - 0.016*(anos-1890)
        + rng.normal(0,0.24,N)
    )
    palavras = np.maximum(140,np.rint(np.exp(log_palavras))).astype(int)

    especiais = rng.choice(N,3,replace=False)
    palavras[especiais] = np.rint(
        palavras[especiais]*rng.uniform(1.65,2.05,3)
    ).astype(int)

    densidade = np.clip(rng.normal(365,38,N),270,470)
    paginas = np.maximum(1,np.rint(palavras/densidade)).astype(int)

    taxa = {"notícia":7.0,"editorial":4.7,"carta":5.8}
    pessoas = rng.poisson(palavras/1000*np.array([taxa[g] for g in generos]))

    df = pd.DataFrame({
        "id_documento":[f"R{i:03d}" for i in range(1,N+1)],
        "ano":anos, "genero":generos, "local":locais, "tema":temas,
        "palavras":palavras, "paginas":paginas, "pessoas":pessoas,
    })
    ARQUIVO.parent.mkdir(exist_ok=True)
    df.to_csv(ARQUIVO,index=False)
    return df

if ARQUIVO.exists():
    dados = pd.read_csv(ARQUIVO)
    print("Dados carregados de", ARQUIVO)
else:
    dados = gerar_dados()
    print("Dados reproduzidos em", ARQUIVO)

print("Dimensões:", dados.shape)
display(dados.head())

Dados reproduzidos em dados/documentos.csv
Dimensões: (120, 8)


,id_documento,ano,genero,local,tema,palavras,paginas,pessoas
0,R001,1906,editorial,Capital,progresso,1458,4,5
1,R002,1903,notícia,Capital,educação,1033,2,5
2,R003,1892,carta,Capital,saúde,1104,3,5
3,R004,1896,notícia,Interior,saúde,764,2,5
4,R005,1906,editorial,Capital,educação,1231,4,9


O notebook usa `dados/documentos.csv` quando ele já existe. Caso contrário, reproduz o mesmo mecanismo gerador do guia da Unidade 5. Isso mantém a análise coerente com o conjunto ampliado de 120 registros.

In [ ]:
resumo = dados.groupby("local")["palavras"].agg(
    ["count","mean","median","std","min","max"]
)
display(resumo.round(2))

capital = dados.loc[dados["local"].eq("Capital"),"palavras"]
interior = dados.loc[dados["local"].eq("Interior"),"palavras"]

plt.figure(figsize=(8,5))
plt.boxplot([capital,interior],tick_labels=["Capital","Interior"])
plt.xlabel("Grupo")
plt.ylabel("Número de palavras")
plt.title("Distribuição da extensão documental por local")
plt.grid(axis="y",alpha=0.2)
plt.show()

A tabela resume os grupos; o gráfico mostra sua distribuição. Uma diferença de médias não elimina a **heterogeneidade interna** nem implica separação completa entre os grupos.

Uma propriedade agregada dos grupos não deve ser convertida automaticamente em uma afirmação sobre todos os indivíduos que os compõem.

## 2. Médias, medianas e proporções

Compararemos três propriedades:

1. média do número de palavras;
2. mediana do número de palavras;
3. proporção de documentos cujo `tema` é `educação`.

A proporção não mede quanto cada texto discute educação; mede apenas a frequência da categoria `educação` dentro de cada grupo.

In [ ]:
por_local = dados.groupby("local")
medidas = pd.DataFrame({
    "media_palavras": por_local["palavras"].mean(),
    "mediana_palavras": por_local["palavras"].median(),
    "proporcao_educacao": por_local["tema"].apply(lambda s: s.eq("educação").mean()),
})
diferencas = medidas.loc["Capital"] - medidas.loc["Interior"]

display(medidas.round(3))
display(diferencas.rename("Capital menos Interior").round(3))

Diferenças de médias e medianas são lidas em **palavras**. Diferenças de proporções podem ser lidas como proporções ou, multiplicadas por 100, como **pontos percentuais**.

A escolha da medida deve preceder a observação do resultado.

## 3. Diferença absoluta e diferença relativa

A diferença absoluta preserva a escala original:

\[
\Delta=\hat\theta_A-\hat\theta_B.
\]

Tomando \(B\) como referência, a diferença relativa é

\[
\Delta_{rel}=
\frac{\hat\theta_A-\hat\theta_B}{\hat\theta_B}.
\]

A diferença relativa depende da referência escolhida e pode se tornar instável quando o denominador se aproxima de zero. Por isso, deve ser apresentada junto com a diferença absoluta.

In [ ]:
media_capital = medidas.loc["Capital","media_palavras"]
media_interior = medidas.loc["Interior","media_palavras"]

dif_abs = media_capital - media_interior
dif_rel = dif_abs / media_interior

pd.Series({
    "diferença absoluta (palavras)": dif_abs,
    "diferença relativa a Interior": dif_rel,
}).round(3)

## 4. Tamanho de efeito

A diferença bruta responde: **quantas palavras separam, em média, os grupos?**

O tamanho de efeito padronizado responde a outra pergunta: **quão grande é essa diferença em relação à variabilidade dentro dos grupos?**

\[
d=\frac{\bar{x}_A-\bar{x}_B}{s_p},
\qquad
s_p=
\sqrt{
\frac{(n_A-1)s_A^2+(n_B-1)s_B^2}
{n_A+n_B-2}
}.
\]

O valor de \(d\) expressa a diferença em unidades de desvio-padrão combinado.

**Tamanho de efeito não é sinônimo de relevância substantiva.** A relevância depende da pergunta, da escala e do contexto. Rótulos automáticos como “pequeno”, “médio” ou “grande” não substituem essa interpretação.

In [ ]:
desvio_combinado = math.sqrt(
    ((len(capital)-1)*capital.var(ddof=1)
     + (len(interior)-1)*interior.var(ddof=1))
    / (len(capital)+len(interior)-2)
)

d_padronizado = (capital.mean()-interior.mean())/desvio_combinado

pd.Series({
    "diferença bruta": capital.mean()-interior.mean(),
    "desvio-padrão combinado": desvio_combinado,
    "d padronizado": d_padronizado,
}).round(3)

## 5. Sensibilidade e casos influentes

No material original havia um caso extremo conhecido de antemão. No conjunto ampliado, é melhor não fixar um identificador. Vamos localizar o maior documento e observar quanto a diferença entre grupos muda quando ele é retirado.

Isso **não significa que o caso deva ser excluído**. A retirada é apenas um experimento de sensibilidade.

In [ ]:
idx = dados["palavras"].idxmax()
caso = dados.loc[idx]
sem_caso = dados.drop(index=idx)

dif_sem_caso = (
    sem_caso.loc[sem_caso["local"].eq("Capital"),"palavras"].mean()
    - sem_caso.loc[sem_caso["local"].eq("Interior"),"palavras"].mean()
)

display(pd.DataFrame({
    "comparação":["todos os registros","sem o maior documento"],
    "Capital - Interior":[dif_abs,dif_sem_caso],
}).round(2))

print("Caso influente usado no teste:")
display(caso.to_frame("valor"))

Se a estimativa muda muito, aprendemos que a conclusão é sensível àquele registro. Se muda pouco, aprendemos que a diferença agregada não depende fortemente dele.

A sensibilidade é uma propriedade da análise; decidir se um caso é erro, exceção legítima ou documento substantivamente importante exige voltar à fonte.

In [ ]:
casos_extremos = pd.concat([
    dados.nsmallest(3,"palavras"),
    dados.nlargest(3,"palavras"),
]).drop_duplicates()

display(
    casos_extremos[[
        "id_documento","ano","genero","local","tema",
        "palavras","paginas","pessoas"
    ]].sort_values("palavras")
)

## 6. Relevância substantiva

Até aqui calculamos diferenças, proporções, tamanho de efeito e sensibilidade. Nenhuma dessas medidas, isoladamente, responde se a diferença é **importante** para a pergunta de pesquisa.

Neste conjunto didático, `palavras` representa extensão documental. Uma diferença de centenas de palavras pode ser importante para uma pergunta sobre extensão, mas não demonstra diferença de conteúdo, qualidade, influência histórica ou intensidade temática.

A distinção central é:

\[
\text{diferença observada}
\neq
\text{tamanho de efeito}
\neq
\text{relevância substantiva}.
\]

No próximo notebook acrescentaremos a **incerteza**. Ela também não substitui nenhuma dessas dimensões.

## U05-A02 — Atividade integrada — quadro de estimativas

1. escolha um critério de agrupamento e uma propriedade coerente com sua pergunta;
2. apresente a distribuição e ao menos duas medidas pertinentes;
3. calcule uma diferença absoluta e, quando fizer sentido, relativa ou padronizada;
4. identifique ao menos dois casos que ajudem a interpretar o resultado;
5. faça uma análise simples de sensibilidade;
6. escreva separadamente:
   - o que foi calculado;
   - o que o resultado permite afirmar;
   - o que ele não permite afirmar;
   - por que a diferença pode ou não ser substantivamente relevante.

**Agrupamento e propriedade:** Escreva aqui.

**Estimativas e diferenças:** Escreva aqui.

**Casos inspecionados:** Escreva aqui.

**Sensibilidade:** Escreva aqui.

**Relevância substantiva e limites:** Escreva aqui.